# Analyse RH — Turnover & Performance
Exploration d'un dataset fictif de 5 000 employés pour identifier les leviers de rétention.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
n = 5000
departements = ['Support client', 'Commercial', 'RH', 'Finance', 'IT', 'Marketing', 'Operations']
niveaux = ['Junior', 'Confirmé', 'Senior', 'Manager']

df = pd.DataFrame({
    'employe_id': range(1, n + 1),
    'departement': np.random.choice(departements, n, p=[0.25, 0.20, 0.10, 0.10, 0.15, 0.10, 0.10]),
    'niveau': np.random.choice(niveaux, n, p=[0.35, 0.30, 0.25, 0.10]),
    'anciennete_ans': np.random.exponential(scale=4, size=n).clip(0, 20).round(1),
    'salaire': np.random.normal(42000, 12000, n).clip(24000, 95000).round(-2),
    'score_performance': np.random.normal(6.2, 2.1, n).clip(1, 10).round(1),
    'jours_absence': np.random.poisson(lam=7, size=n).clip(0, 40),
    'a_quitte': np.random.choice([0, 1], n, p=[0.857, 0.143]),
})
mask = df['score_performance'] < 3
df.loc[mask, 'a_quitte'] = np.random.choice([0, 1], mask.sum(), p=[0.40, 0.60])

df.head()

## Taux de turnover global

In [ ]:
taux = df['a_quitte'].mean() * 100
print(f"Taux de turnover global : {taux:.1f}%")

## Turnover par département

In [ ]:
turnover_dept = df.groupby('departement')['a_quitte'].mean().sort_values(ascending=False) * 100

plt.figure(figsize=(10, 5))
sns.barplot(x=turnover_dept.index, y=turnover_dept.values, palette='Blues_d')
plt.title('Taux de turnover par département (%)')
plt.ylabel('Turnover (%)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## Absentéisme par département

In [ ]:
absence_dept = df.groupby('departement')['jours_absence'].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=absence_dept.index, y=absence_dept.values, palette='Oranges_d')
plt.title("Jours d'absence moyens par département")
plt.ylabel('Jours absence')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## Performance vs Turnover

In [ ]:
df['score_bin'] = pd.cut(df['score_performance'], bins=[0,3,5,7,10], labels=['< 3','3-5','5-7','> 7'])
turnover_perf = df.groupby('score_bin')['a_quitte'].mean() * 100

plt.figure(figsize=(8, 5))
sns.barplot(x=turnover_perf.index, y=turnover_perf.values, palette='Reds_d')
plt.title('Turnover selon le score de performance')
plt.ylabel('Turnover (%)')
plt.xlabel('Score de performance')
plt.tight_layout()
plt.show()

## Conclusion

- Turnover global à **14,3%** — concentré sur Support client et Commercial
- Les employés avec score < 3 partent **3x plus** que les bons performers
- Support client cumule le plus d'absences ET le plus fort turnover
- Recommandation : cibler les actions RH sur Support client en priorité